In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import json
import time

def setup_driver():
    """Selenium driver beállítása"""
    chrome_options = Options()
    chrome_options.add_argument("--headless")  # Háttérben futtatás
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--window-size=1920,1080")
    
    driver = webdriver.Chrome(options=chrome_options)
    return driver

def scrape_with_selenium(url):
    """Scraping Seleniummal"""
    driver = None
    try:
        print("1. Driver indítása...")
        driver = setup_driver()
        
        print("2. Oldal betöltése...")
        driver.get(url)
        
        # Várakozás a tartalom betöltésére
        print("3. Várakozás a tartalom betöltésére...")
        wait = WebDriverWait(driver, 20)
        
        # Várjuk meg, hogy a fő elemek betöltődjenek
        wait.until(EC.presence_of_element_located((By.CLASS_NAME, "status-live-game-card-content")))
        
        print("4. Tartalom betöltve, HTML kinyerése...")
        # Kis várakozás a teljes betöltésre
        time.sleep(3)
        
        # HTML kinyerése
        page_source = driver.page_source
        
        # HTML mentése diagnosztikához
        with open('selenium_page_content.html', 'w', encoding='utf-8') as f:
            f.write(page_source)
        print("✅ Selenium HTML elmentve: selenium_page_content.html")
        
        # BeautifulSoup parsing
        soup = BeautifulSoup(page_source, 'html.parser')
        
        # Eredmény objektum
        result = {
            'blue_team': {},
            'red_team': {},
            'players': []
        }
        
        # Team statisztikák kinyerése
        team_stats = extract_team_stats(soup)
        result['blue_team'] = team_stats['blue']
        result['red_team'] = team_stats['red']
        
        # Player statisztikák kinyerése
        result['players'] = extract_player_stats(soup)
        
        return result
        
    except Exception as e:
        print(f"❌ Hiba: {str(e)}")
        return {'error': str(e)}
    finally:
        if driver:
            driver.quit()
            print("✅ Driver leállítva")

def extract_team_stats(soup):
    """Kinyeri a csapat statisztikákat"""
    stats = {'blue': {}, 'red': {}}
    
    print("5. Team statisztikák kinyerése...")
    
    # Blue team stats
    blue_team = soup.find('div', class_='blue-team')
    if blue_team:
        print("   ✅ Blue team megtalálva")
        stats['blue'] = {
            'inhibitors': extract_stat_value(blue_team, 'inhibitors'),
            'barons': extract_stat_value(blue_team, 'barons'),
            'towers': extract_stat_value(blue_team, 'towers'),
            'gold': extract_gold_value(blue_team),
            'kills': extract_stat_value(blue_team, 'kills'),
            'dragons': extract_dragons(blue_team)
        }
    else:
        print("   ❌ Blue team NEM található")
    
    # Red team stats
    red_team = soup.find('div', class_='red-team')
    if red_team:
        print("   ✅ Red team megtalálva")
        stats['red'] = {
            'inhibitors': extract_stat_value(red_team, 'inhibitors'),
            'barons': extract_stat_value(red_team, 'barons'),
            'towers': extract_stat_value(red_team, 'towers'),
            'gold': extract_gold_value(red_team),
            'kills': extract_stat_value(red_team, 'kills'),
            'dragons': extract_dragons(red_team)
        }
    else:
        print("   ❌ Red team NEM található")
    
    return stats

def extract_stat_value(team_div, stat_class):
    """Kinyeri egy adott stat értékét"""
    stat_div = team_div.find('div', class_=f'team-stats {stat_class}')
    if stat_div:
        # Az utolsó gyermek elem tartalmazza a számértéket
        children = list(stat_div.children)
        if children:
            last_child = children[-1]
            if last_child.name is None:  # Szöveg node
                return last_child.strip()
            else:
                return last_child.get_text(strip=True)
    return "0"

def extract_gold_value(team_div):
    """Kinyeri a gold értéket (speciális formázás)"""
    gold_div = team_div.find('div', class_='team-stats gold')
    if gold_div:
        span = gold_div.find('span')
        if span:
            return span.get_text(strip=True)
    return "0"

def extract_dragons(team_div):
    """Kinyeri a sárkány típusokat"""
    dragons = []
    dragon_svgs = team_div.find_all('svg', class_='dragon')
    
    for svg in dragon_svgs:
        path = svg.find('path', class_='shape')
        if path:
            fill_color = path.get('fill', '')
            dragon_type = map_dragon_type(fill_color)
            dragons.append(dragon_type)
    
    return dragons

def map_dragon_type(fill_color):
    """Megfelelteti a színt a sárkány típusának"""
    dragon_map = {
        '#A8805D': 'Earth',
        '#67C4B0': 'Ocean',
        '#ADD2ED': 'Cloud',
        '#F0BE1A': 'Infernal',
        '#8C52FF': 'Hextech',
        '#5CD6A9': 'Chemtech'
    }
    return dragon_map.get(fill_color, f'Unknown ({fill_color})')

def extract_player_stats(soup):
    """Kinyeri a játékos statisztikákat"""
    players = []
    
    print("6. Player statisztikák kinyerése...")
    
    # Player rows keresése
    player_rows = soup.find_all('tr', class_='player-stats-row')
    print(f"   Player rows találva: {len(player_rows)}")
    
    for i, row in enumerate(player_rows):
        print(f"   Feldolgozás: {i+1}. játékos")
        player_data = extract_player_row_data(row)
        if player_data:
            players.append(player_data)
    
    return players

def extract_player_row_data(row):
    """Kinyeri az adatokat egy játékos sorból - biztonságos verzió"""
    try:
        # Champion info (ugyanaz marad)
        champion_info = row.find('div', class_='player-champion-info')
        if not champion_info:
            return None
            
        # Champion név, player név, level, health, items (ugyanaz marad)
        champion_name_spans = champion_info.find_all('span')
        champion_name = "Unknown"
        for span in champion_name_spans:
            text = span.get_text(strip=True)
            if text and text not in ['', 'T1', 'TOPESPORTS']:
                champion_name = text
                break
        
        player_name_elem = champion_info.find('span', class_='player-card-player-name')
        player_name = player_name_elem.get_text(strip=True) if player_name_elem else "Unknown"
        
        level_elem = champion_info.find('span', class_='player-champion-info-level')
        level = level_elem.get_text(strip=True) if level_elem else "0"
        
        health_div = row.find('div', class_='mini-health-bar')
        health_text = ""
        if health_div:
            health_span = health_div.find('span', class_='mini-current-health')
            if health_span:
                health_text = health_span.get_text(strip=True)
        
        items_div = row.find('div', class_='player-stats-items')
        items = []
        if items_div:
            item_imgs = items_div.find_all('img')
            for img in item_imgs:
                src = img.get('src', '')
                if 'item/' in src:
                    item_id = src.split('/')[-1].replace('.png', '')
                    items.append(item_id)
        
        # Oszlopok keresése class alapján
        columns = row.find_all('td')
        stats = {
            'champion': champion_name,
            'player_name': player_name,
            'level': level,
            'health': health_text,
            'items': items,
            'cs': "0",
            'kills': "0", 
            'deaths': "0",
            'assists': "0",
            'gold': "0",
            'gold_difference': "0"
        }
        
        # Minden oszlop feldolgozása
        for i, column in enumerate(columns):
            text = extract_column_text([column], 0)  # Csak az aktuális oszlop
            
            # Próbáljuk megállapítani, hogy milyen típusú adat lehet
            if 'player-stats-kda' in str(column):
                # KDA oszlopok - sorrend alapján
                if stats['kills'] == "0":
                    stats['kills'] = text
                elif stats['deaths'] == "0":
                    stats['deaths'] = text
                elif stats['assists'] == "0":
                    stats['assists'] = text
            elif any(char in text for char in ['+', '-']) and any(char in text for char in ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']):
                # Gold difference (tartalmaz + vagy - jelet és számot)
                stats['gold_difference'] = text
            elif text.isdigit() and len(text) <= 4:  # CS (általában 3-4 jegyű szám)
                if stats['cs'] == "0":
                    stats['cs'] = text
            elif any(char in text for char in [',']) and any(char in text for char in ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']):
                # Gold (tartalmaz vesszőt és számot)
                stats['gold'] = text
        
        return stats
        
    except Exception as e:
        print(f"   Hiba a játékos adat kinyerése során: {e}")
        return None

def extract_column_text(columns, index):
    """Kinyeri a szöveget egy adott oszlopból"""
    if len(columns) > index:
        # Különböző lehetséges class-ok
        possible_classes = ['player-stats', 'player-stats-kda']
        for class_name in possible_classes:
            stat_div = columns[index].find('div', class_=class_name)
            if stat_div:
                return stat_div.get_text(strip=True)
        
        # Ha nincs div, akkor közvetlenül a td szövege
        return columns[index].get_text(strip=True)
    return "0"

# Diagnosztikai funkció Seleniummal
def diagnose_selenium(url):
    """Diagnosztika Seleniummal"""
    driver = None
    try:
        driver = setup_driver()
        driver.get(url)
        
        # Várakozás
        wait = WebDriverWait(driver, 20)
        wait.until(EC.presence_of_element_located((By.CLASS_NAME, "status-live-game-card-content")))
        time.sleep(3)
        
        # Elemek keresése
        elements_to_check = [
            'status-live-game-card-content',
            'blue-team', 
            'red-team',
            'player-stats-row'
        ]
        
        print("Selenium diagnosztika:")
        for element_class in elements_to_check:
            elements = driver.find_elements(By.CLASS_NAME, element_class)
            print(f"  {element_class}: {len(elements)} elem")
            
        return True
        
    except Exception as e:
        print(f"Diagnosztika hiba: {e}")
        return False
    finally:
        if driver:
            driver.quit()

# Futtatás
if __name__ == "__main__":
    url = "https://andydanger.github.io/live-lol-esports/#/live/113475871523985235"
    
    print("=" * 50)
    print("SELENIUM SCRAPER INDUL")
    print("=" * 50)
    
    # Először diagnosztika
    print("Diagnosztika futtatása...")
    diagnose_selenium(url)
    
    print("\n" + "=" * 50)
    print("FŐ SCRAPING INDUL")
    print("=" * 50)
    
    # Fő scraping
    result = scrape_with_selenium(url)
    
    print("\n" + "=" * 50)
    print("EREDMÉNY")
    print("=" * 50)
    
    if 'error' not in result:
        print(json.dumps(result, indent=2, ensure_ascii=False))
    else:
        print(f"Hiba: {result['error']}")